In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import numpy as np
import pandas as pd
from pathlib import Path
from collections import OrderedDict
from sklearn.metrics import f1_score, precision_score, recall_score
import copy
import os
import glob

In [6]:
FINAL_CLASSES = ["AF", "IAVB", "LAD", "LBBB", "NSIVCB", "NSR", "PAC", "QAb", "RBBB", "SB", "STach", "TAb"]
PROCESSED_DIR = r"C:\ECG_Project\training\processed_signals"
SPLIT_CSV_PATH = r"C:\ECG_Project\training\All excel files\ecg_metadata_train_test_split.csv"
FEDERATED_CLIENTS = ["chapman_shaoxing", "cpsc_2018", "georgia", "ningbo", "ptb-xl"]
metadata_df = pd.read_csv(SPLIT_CSV_PATH)
CHECKPOINT_DIR = r"C:\ECG_Project\training"

In [17]:
class ResidualBlock1D(nn.Module):
    """
    One residual block: two conv layers + a skip connection.
    If input/output channels differ, or stride > 1, the skip connection
    uses a 1x1 conv to match dimensions.
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = F.relu(out)
        return out


class ResNet1D34(nn.Module):
    """
    1D ResNet-34, adapted for 12-lead ECG classification.
    stem -> 4 stages of residual blocks -> global pool -> classifier head
    """
    def __init__(self, in_channels=12, num_classes=12):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )

        # ResNet-34 recipe: [3, 4, 6, 3] blocks per stage
        self.layer1 = self._make_layer(64, 64, num_blocks=3, stride=1)
        self.layer2 = self._make_layer(64, 128, num_blocks=4, stride=2)
        self.layer3 = self._make_layer(128, 256, num_blocks=6, stride=2)
        self.layer4 = self._make_layer(256, 512, num_blocks=3, stride=2)

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = [ResidualBlock1D(in_channels, out_channels, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock1D(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.global_pool(x)
        x = x.squeeze(-1)
        x = self.fc(x)
        return x


if __name__ == "__main__":
    # Sanity check with dummy data — run this file directly to verify the architecture
    model = ResNet1D34(in_channels=12, num_classes=12)

    dummy_input = torch.randn(4, 12, 5000)
    output = model(dummy_input)

    total_params = sum(p.numel() for p in model.parameters())

    print("Output shape:", output.shape)          # expect torch.Size([4, 12])
    print(f"Total parameters: {total_params:,}")  # expect ~7,229,324
# ============ END: resnet1d.py ============

Output shape: torch.Size([4, 12])
Total parameters: 7,229,324


In [19]:
class ECGDataset(Dataset):
    """
    Loads preprocessed (12, 5000) .npy signals + multi-label targets
    for one specific hospital client and split (train/test).

    Defensive by design: filters out any record whose .npy file doesn't
    actually exist on disk, at __init__ time — not mid-training. This is
    the fix for the FileNotFoundError bug from Stage 4; baking it in here
    means it can never resurface, even if a metadata CSV gets stale again.
    """
    def __init__(self, metadata_df: pd.DataFrame, processed_dir: str,
                 source_hospital: str, split: str):
        candidate_df = metadata_df[
            (metadata_df["source_hospital"] == source_hospital) &
            (metadata_df["split"] == split)
        ].reset_index(drop=True)

        exists_mask = candidate_df["record_id"].apply(
            lambda rid: Path(f"{processed_dir}/{rid}.npy").exists()
        )
        dropped = (~exists_mask).sum()
        if dropped > 0:
            print(f"  [{source_hospital}/{split}] Skipping {dropped} records with missing .npy files")

        self.df = candidate_df[exists_mask].reset_index(drop=True)
        self.processed_dir = processed_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        record_id = row["record_id"]

        signal = np.load(f"{self.processed_dir}/{record_id}.npy")
        label = row[FINAL_CLASSES].values.astype(np.float32)

        return torch.tensor(signal, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)


def build_client_loaders(metadata_df: pd.DataFrame, federated_clients: list,
                          processed_dir: str = PROCESSED_DIR, batch_size: int = 16):
    """
    Builds one train + test DataLoader pair per hospital client.
    Returns a dict: { hospital_name: {"train": loader, "test": loader, "train_size": int} }
    """
    client_loaders = {}
    for hospital in federated_clients:
        train_ds = ECGDataset(metadata_df, processed_dir, hospital, "train")
        test_ds = ECGDataset(metadata_df, processed_dir, hospital, "test")

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

        client_loaders[hospital] = {
            "train": train_loader,
            "test": test_loader,
            "train_size": len(train_ds),
        }
        print(f"{hospital}: {len(train_ds)} train / {len(test_ds)} test")

    return client_loaders


if __name__ == "__main__":

    FEDERATED_CLIENTS = ["chapman_shaoxing", "cpsc_2018", "georgia", "ningbo", "ptb-xl"]

    client_loaders = build_client_loaders(metadata_df, FEDERATED_CLIENTS)

    # End-to-end sanity check: pull one real batch through the model
    model = ResNet1D34(in_channels=12, num_classes=12)

    test_hospital = "georgia"
    batch_signals, batch_labels = next(iter(client_loaders[test_hospital]["train"]))
    print(f"\nBatch signals shape: {batch_signals.shape}")   # expect (16, 12, 5000)
    print(f"Batch labels shape: {batch_labels.shape}")       # expect (16, 12)

    test_output = model(batch_signals)
    print(f"Model output shape on real batch: {test_output.shape}")   # expect (16, 12)
# ============ END: ecg_dataset.py ============

chapman_shaoxing: 7522 train / 1885 test
cpsc_2018: 4794 train / 1200 test
georgia: 6520 train / 1641 test
ningbo: 22246 train / 5594 test
ptb-xl: 16829 train / 4203 test

Batch signals shape: torch.Size([16, 12, 5000])
Batch labels shape: torch.Size([16, 12])
Model output shape on real batch: torch.Size([16, 12])


In [20]:
def get_model_parameters(model):
    """Extract model weights as a plain state_dict (for aggregation)."""
    return model.state_dict()


def set_model_parameters(model, state_dict):
    """Load a state_dict back into the model."""
    model.load_state_dict(state_dict, strict=True)


def train_one_client(model, train_loader, device, epochs=1, lr=1e-3):
    model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0
        for signals, labels in train_loader:
            signals, labels = signals.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(signals)
            loss = criterion(outputs, labels)

            if not torch.isfinite(loss):
                print("  Skipping a batch with non-finite loss")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)
    return avg_loss


def evaluate_one_client(model, test_loader, device):
    model.to(device)
    model.eval()
    criterion = nn.BCEWithLogitsLoss()

    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for signals, labels in test_loader:
            signals, labels = signals.to(device), labels.to(device)
            outputs = model(signals)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).float()
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    micro_precision = precision_score(all_labels, all_preds, average="micro", zero_division=0)
    micro_recall = recall_score(all_labels, all_preds, average="micro", zero_division=0)

    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)      # NEW
    macro_precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)  # NEW
    macro_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)        # NEW

    return avg_loss, micro_f1, micro_precision, micro_recall, macro_f1, macro_precision, macro_recall

In [9]:
print(ResNet1D34)
print(build_client_loaders)
print(get_model_parameters, set_model_parameters, train_one_client, evaluate_one_client)

<class '__main__.ResNet1D34'>
<function build_client_loaders at 0x0000017D35867600>
<function get_model_parameters at 0x0000017D3655ACA0> <function set_model_parameters at 0x0000017D3655AA20> <function train_one_client at 0x0000017D3A78C4A0> <function evaluate_one_client at 0x0000017D3A78E480>


In [33]:
# ============ START: fedavg.py ============

NUM_ROUNDS = 2       # SMOKE TEST first — raise this only after this runs clean
LOCAL_EPOCHS = 1
BATCH_SIZE = 16


def federated_average(client_state_dicts, client_sizes):
    """
    Weighted average of multiple clients' model weights.
    client_state_dicts: list of state_dicts (one per client, post-local-training)
    client_sizes: list of ints (train set size per client) — a hospital with more
                  data gets proportionally more influence, standard FedAvg (McMahan et al.)
    """
    total_size = sum(client_sizes)
    avg_weights = OrderedDict()

    for key in client_state_dicts[0].keys():
        weighted_sum = sum(
            client_state_dicts[i][key].float() * (client_sizes[i] / total_size)
            for i in range(len(client_state_dicts))
        )
        avg_weights[key] = weighted_sum

    return avg_weights


def run_federated_training(client_loaders, device, num_rounds=NUM_ROUNDS, local_epochs=LOCAL_EPOCHS):
    global_model = ResNet1D34(in_channels=12, num_classes=12).to(device)
    round_history = []

    for round_num in range(1, num_rounds + 1):
        print(f"\n{'='*50}\nROUND {round_num}/{num_rounds}\n{'='*50}")

        client_state_dicts = []
        client_sizes = []
        round_metrics = {}

        global_weights = copy.deepcopy(global_model.state_dict())

        for hospital in FEDERATED_CLIENTS:
            local_model = ResNet1D34(in_channels=12, num_classes=12).to(device)
            set_model_parameters(local_model, global_weights)

            train_loss = train_one_client(
                local_model, client_loaders[hospital]["train"], device, epochs=local_epochs
            )

            test_loss, f1, precision, recall = evaluate_one_client(
                local_model, client_loaders[hospital]["test"], device
            )

            print(f"  [{hospital}] train_loss={train_loss:.4f} test_loss={test_loss:.4f} "
                  f"micro_f1={f1:.4f} precision={precision:.4f} recall={recall:.4f}")

            round_metrics[hospital] = {
                "train_loss": train_loss, "test_loss": test_loss,
                "micro_f1": f1, "precision": precision, "recall": recall,
            }

            client_state_dicts.append(get_model_parameters(local_model))
            client_sizes.append(client_loaders[hospital]["train_size"])

        new_global_weights = federated_average(client_state_dicts, client_sizes)
        set_model_parameters(global_model, new_global_weights)

        round_history.append(round_metrics)
        print(f"\nRound {round_num} complete. Global model updated.")

    return global_model, round_history


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    metadata_df = pd.read_csv(SPLIT_CSV_PATH)
    client_loaders = build_client_loaders(metadata_df, FEDERATED_CLIENTS, batch_size=BATCH_SIZE)

    global_model, round_history = run_federated_training(client_loaders, device)

    print("\n\nFederated training smoke test complete.")

    # Save the trained global model + round history so nothing is lost if the kernel restarts
    torch.save(global_model.state_dict(), r"C:\ECG_Project\training\global_model_checkpoint.pt")

    history_rows = []
    for round_num, metrics in enumerate(round_history, start=1):
        for hospital, vals in metrics.items():
            history_rows.append({"round": round_num, "hospital": hospital, **vals})
    pd.DataFrame(history_rows).to_csv(r"C:\ECG_Project\training\fedavg_round_history.csv", index=False)
    print("Saved: global_model_checkpoint.pt and fedavg_round_history.csv")
# ============ END: fedavg.py ============

Using device: cuda
chapman_shaoxing: 7522 train / 1885 test
cpsc_2018: 4794 train / 1200 test
georgia: 6520 train / 1641 test
ningbo: 22246 train / 5594 test
ptb-xl: 16829 train / 4203 test

ROUND 1/2
  [chapman_shaoxing] train_loss=0.1798 test_loss=0.1447 micro_f1=0.7839 precision=0.8680 recall=0.7146
  [cpsc_2018] train_loss=0.1683 test_loss=0.1596 micro_f1=0.6125 precision=0.6470 recall=0.5815
  [georgia] train_loss=0.2447 test_loss=0.2586 micro_f1=0.4872 precision=0.6644 recall=0.3846
  [ningbo] train_loss=0.1265 test_loss=0.1545 micro_f1=0.7649 precision=0.8168 recall=0.7192
  [ptb-xl] train_loss=0.1557 test_loss=0.1368 micro_f1=0.7966 precision=0.8777 recall=0.7292

Round 1 complete. Global model updated.

ROUND 2/2
  [chapman_shaoxing] train_loss=0.1245 test_loss=0.1239 micro_f1=0.8230 precision=0.8147 recall=0.8314
  [cpsc_2018] train_loss=0.1305 test_loss=0.1210 micro_f1=0.7529 precision=0.8513 recall=0.6748
  [georgia] train_loss=0.2000 test_loss=0.1923 micro_f1=0.6360 precis

In [ ]:
processed_dir = Path(r"C:\ECG_Project\training\processed_signals")

# Master list to collect all corrupt records across all clients
all_bad_records = []

# Loop through each client to find NaN/Inf files
for client in FEDERATED_CLIENTS:
    # Filter record IDs belonging to the current client
    client_ids = metadata_df[metadata_df["source_hospital"] == client]["record_id"]
    client_bad_records = []
    
    for rid in client_ids:
        file_path = processed_dir / f"{rid}.npy"
        
        # Safe check in case a file is missing in the directory
        if file_path.exists():
            sig = np.load(file_path)
            if not np.isfinite(sig).all():
                client_bad_records.append(rid)
        else:
            client_bad_records.append(rid)  # Treat missing files as bad records
            
    print(f"Found {len(client_bad_records)} {client} records with NaN/Inf values")
    if client_bad_records:
        print(f"Sample: {client_bad_records[:10]}")
        
    # Append the client's bad records to the master list
    all_bad_records.extend(client_bad_records)

# Print total findings
print(f"\nTotal bad records found across all clients: {len(all_bad_records)}")

# Filter out the bad records from the metadata DataFrame
metadata_df = metadata_df[~metadata_df["record_id"].isin(all_bad_records)].reset_index(drop=True)
print(f"Dropped bad records. New metadata shape: {metadata_df.shape}")

# Save the updated dataset
metadata_df.to_csv(metadata_path, index=False)
print("Saved updated split CSV successfully!")

In [14]:
# ============ START: fedavg.py (30-round run, resumable) ============
NUM_ROUNDS = 30
LOCAL_EPOCHS = 1
BATCH_SIZE = 16
CHECKPOINT_EVERY = 5  # save + allow resume every 5 rounds


def federated_average(client_state_dicts, client_sizes):
    """
    Weighted average of multiple clients' model weights.
    A hospital with more training data gets proportionally more influence
    (standard FedAvg, McMahan et al.).
    """
    total_size = sum(client_sizes)
    avg_weights = OrderedDict()
    for key in client_state_dicts[0].keys():
        weighted_sum = sum(
            client_state_dicts[i][key].float() * (client_sizes[i] / total_size)
            for i in range(len(client_state_dicts))
        )
        avg_weights[key] = weighted_sum
    return avg_weights


def find_latest_checkpoint(checkpoint_dir):
    """Finds the highest-round checkpoint saved so far, if any."""
    pattern = os.path.join(checkpoint_dir, "global_model_round*.pt")
    checkpoints = glob.glob(pattern)
    if not checkpoints:
        return None, 0
    rounds = [int(f.split("round")[-1].split(".pt")[0]) for f in checkpoints]
    latest_round = max(rounds)
    latest_path = os.path.join(checkpoint_dir, f"global_model_round{latest_round}.pt")
    return latest_path, latest_round


def run_federated_training(client_loaders, device, num_rounds=NUM_ROUNDS, local_epochs=LOCAL_EPOCHS,
                            checkpoint_dir=CHECKPOINT_DIR, checkpoint_every=CHECKPOINT_EVERY):
    global_model = ResNet1D34(in_channels=12, num_classes=12).to(device)

    # RESUME LOGIC: pick up from the last saved round instead of starting over
    latest_ckpt, start_round = find_latest_checkpoint(checkpoint_dir)
    round_history = []

    if latest_ckpt:
        print(f"Resuming from checkpoint: {latest_ckpt} (round {start_round})")
        global_model.load_state_dict(torch.load(latest_ckpt, map_location=device))

        history_path = os.path.join(checkpoint_dir, "fedavg_round_history.csv")
        if os.path.exists(history_path):
            existing = pd.read_csv(history_path)
            for r in sorted(existing["round"].unique()):
                round_history.append(
                    existing[existing["round"] == r].set_index("hospital")[
                        ["train_loss", "test_loss", "micro_f1", "precision", "recall"]
                    ].to_dict("index")
                )
    else:
        print("No checkpoint found — starting fresh from round 1")

    if start_round >= num_rounds:
        print(f"Already completed {start_round} rounds (target was {num_rounds}). Nothing to do.")
        return global_model, round_history

    for round_num in range(start_round + 1, num_rounds + 1):
        print(f"\n{'='*50}\nROUND {round_num}/{num_rounds}\n{'='*50}")

        client_state_dicts = []
        client_sizes = []
        round_metrics = {}

        global_weights = copy.deepcopy(global_model.state_dict())

        for hospital in FEDERATED_CLIENTS:
            local_model = ResNet1D34(in_channels=12, num_classes=12).to(device)
            set_model_parameters(local_model, global_weights)

            train_loss = train_one_client(
                local_model, client_loaders[hospital]["train"], device, epochs=local_epochs
            )
            test_loss, f1, precision, recall = evaluate_one_client(
                local_model, client_loaders[hospital]["test"], device
            )

            print(f"  [{hospital}] train_loss={train_loss:.4f} test_loss={test_loss:.4f} "
                  f"micro_f1={f1:.4f} precision={precision:.4f} recall={recall:.4f}")

            round_metrics[hospital] = {
                "train_loss": train_loss, "test_loss": test_loss,
                "micro_f1": f1, "precision": precision, "recall": recall,
            }
            client_state_dicts.append(get_model_parameters(local_model))
            client_sizes.append(client_loaders[hospital]["train_size"])

        new_global_weights = federated_average(client_state_dicts, client_sizes)
        set_model_parameters(global_model, new_global_weights)
        round_history.append(round_metrics)
        print(f"\nRound {round_num} complete. Global model updated.")

        # Save checkpoint + history every N rounds, and always on the final round
        if round_num % checkpoint_every == 0 or round_num == num_rounds:
            ckpt_path = os.path.join(checkpoint_dir, f"global_model_round{round_num}.pt")
            torch.save(global_model.state_dict(), ckpt_path)

            history_rows = []
            for r, metrics in enumerate(round_history, start=1):
                for hospital, vals in metrics.items():
                    history_rows.append({"round": r, "hospital": hospital, **vals})
            pd.DataFrame(history_rows).to_csv(
                os.path.join(checkpoint_dir, "fedavg_round_history.csv"), index=False
            )
            print(f"  Checkpoint + history saved at round {round_num}")

    return global_model, round_history


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    client_loaders = build_client_loaders(metadata_df, FEDERATED_CLIENTS, batch_size=BATCH_SIZE)
    global_model, round_history = run_federated_training(client_loaders, device)
    print("\n\nFederated training run complete.")
    print(f"Final checkpoint and history are already saved in {CHECKPOINT_DIR}")
# ============ END: fedavg.py ============

Using device: cuda
chapman_shaoxing: 7522 train / 1885 test
cpsc_2018: 4794 train / 1200 test
georgia: 6520 train / 1641 test
ningbo: 22246 train / 5594 test
ptb-xl: 16829 train / 4203 test
Resuming from checkpoint: C:\ECG_Project\training\global_model_round10.pt (round 10)

ROUND 11/30
  [chapman_shaoxing] train_loss=0.0775 test_loss=0.0746 micro_f1=0.8741 precision=0.8967 recall=0.8527
  [cpsc_2018] train_loss=0.0954 test_loss=0.0759 micro_f1=0.8254 precision=0.8764 recall=0.7801
  [georgia] train_loss=0.1464 test_loss=0.1412 micro_f1=0.7550 precision=0.8224 recall=0.6978
  [ningbo] train_loss=0.0655 test_loss=0.0754 micro_f1=0.8583 precision=0.8806 recall=0.8372
  [ptb-xl] train_loss=0.1026 test_loss=0.1006 micro_f1=0.8500 precision=0.8791 recall=0.8227

Round 11 complete. Global model updated.

ROUND 12/30
  [chapman_shaoxing] train_loss=0.0760 test_loss=0.0756 micro_f1=0.8712 precision=0.8838 recall=0.8589
  [cpsc_2018] train_loss=0.0957 test_loss=0.0742 micro_f1=0.8415 precision=

In [11]:
CHECKPOINT_ROUNDS = [5, 10, 15, 20, 25, 30]   # every round you actually saved a checkpoint for

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def evaluate_checkpoint(model, test_loader, device):
    """Runs one hospital's test set through the model. Pure evaluation — no training."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for signals, labels in test_loader:
            signals = signals.to(device)
            outputs = model(signals)
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    return {
        "micro_f1": f1_score(all_labels, all_preds, average="micro", zero_division=0),
        "macro_f1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "micro_precision": precision_score(all_labels, all_preds, average="micro", zero_division=0),
        "macro_precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "micro_recall": recall_score(all_labels, all_preds, average="micro", zero_division=0),
        "macro_recall": recall_score(all_labels, all_preds, average="macro", zero_division=0),
    }


all_results = []

for round_num in CHECKPOINT_ROUNDS:
    ckpt_path = f"{CHECKPOINT_DIR}\\global_model_round{round_num}.pt"
    print(f"\n--- Evaluating checkpoint: round {round_num} ---")

    model = ResNet1D34(in_channels=12, num_classes=12).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))

    for hospital in FEDERATED_CLIENTS:
        metrics = evaluate_checkpoint(model, client_loaders[hospital]["test"], device)
        print(f"  {hospital}: micro_f1={metrics['micro_f1']:.4f}  macro_f1={metrics['macro_f1']:.4f}")

        all_results.append({
            "round": round_num,
            "hospital": hospital,
            **metrics
        })

results_df = pd.DataFrame(all_results)
out_path = f"{CHECKPOINT_DIR}\\fedavg_micro_macro_full_history.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(results_df)


--- Evaluating checkpoint: round 5 ---
  chapman_shaoxing: micro_f1=0.6988  macro_f1=0.4170
  cpsc_2018: micro_f1=0.3100  macro_f1=0.2296
  georgia: micro_f1=0.5922  macro_f1=0.4062
  ningbo: micro_f1=0.7559  macro_f1=0.4182
  ptb-xl: micro_f1=0.6834  macro_f1=0.3505

--- Evaluating checkpoint: round 10 ---
  chapman_shaoxing: micro_f1=0.7362  macro_f1=0.4690
  cpsc_2018: micro_f1=0.3843  macro_f1=0.2769
  georgia: micro_f1=0.5907  macro_f1=0.4547
  ningbo: micro_f1=0.7430  macro_f1=0.4245
  ptb-xl: micro_f1=0.7168  macro_f1=0.4022

--- Evaluating checkpoint: round 15 ---
  chapman_shaoxing: micro_f1=0.7719  macro_f1=0.5123
  cpsc_2018: micro_f1=0.3814  macro_f1=0.2825
  georgia: micro_f1=0.6135  macro_f1=0.4780
  ningbo: micro_f1=0.7721  macro_f1=0.4566
  ptb-xl: micro_f1=0.7233  macro_f1=0.4238

--- Evaluating checkpoint: round 20 ---
  chapman_shaoxing: micro_f1=0.7984  macro_f1=0.5597
  cpsc_2018: micro_f1=0.4095  macro_f1=0.3060
  georgia: micro_f1=0.6204  macro_f1=0.4931
  ningb

In [7]:
#============FED PROX===========
NUM_ROUNDS = 30          # SAME budget as FedAvg — required for a fair comparison
LOCAL_EPOCHS = 1
BATCH_SIZE = 16
CHECKPOINT_EVERY = 5
MU = 0.001                # FedProx proximal term strength — standard starting value (Li et al., 2020)


def train_one_client_fedprox(model, global_weights, train_loader, device, epochs=1, lr=1e-3, mu=MU):
    """
    Same as train_one_client, PLUS a proximal term that penalizes the local
    model for drifting too far from the global weights it started this round with.
    This is the ONLY difference between FedAvg and FedProx.
    """
    model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    # Keep a frozen reference copy of the starting (global) weights for this round
    global_params = [p.clone().detach() for p in model.parameters()]

    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0
        for signals, labels in train_loader:
            signals, labels = signals.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(signals)
            loss = criterion(outputs, labels)

            # --- Proximal term: mu/2 * ||local_weights - global_weights||^2 ---
            proximal_term = 0.0
            for local_p, global_p in zip(model.parameters(), global_params):
                proximal_term += (local_p - global_p).norm(2) ** 2
            loss = loss + (mu / 2) * proximal_term
            # --------------------------------------------------------------

            if not torch.isfinite(loss):
                print("  Skipping a batch with non-finite loss")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)
    return avg_loss


def federated_average(client_state_dicts, client_sizes):
    total_size = sum(client_sizes)
    avg_weights = OrderedDict()
    for key in client_state_dicts[0].keys():
        weighted_sum = sum(
            client_state_dicts[i][key].float() * (client_sizes[i] / total_size)
            for i in range(len(client_state_dicts))
        )
        avg_weights[key] = weighted_sum
    return avg_weights


def find_latest_checkpoint(checkpoint_dir, prefix="fedprox_global_model_round"):
    pattern = os.path.join(checkpoint_dir, f"{prefix}*.pt")
    checkpoints = glob.glob(pattern)
    if not checkpoints:
        return None, 0
    rounds = [int(f.split("round")[-1].split(".pt")[0]) for f in checkpoints]
    latest_round = max(rounds)
    latest_path = os.path.join(checkpoint_dir, f"{prefix}{latest_round}.pt")
    return latest_path, latest_round


def run_fedprox_training(client_loaders, device, num_rounds=NUM_ROUNDS, local_epochs=LOCAL_EPOCHS,
                          checkpoint_dir=CHECKPOINT_DIR, checkpoint_every=CHECKPOINT_EVERY, mu=MU):
    global_model = ResNet1D34(in_channels=12, num_classes=12).to(device)

    latest_ckpt, start_round = find_latest_checkpoint(checkpoint_dir)
    round_history = []

    if latest_ckpt:
        print(f"Found checkpoint: {latest_ckpt} (round {start_round}) — attempting to load")
        try:
            global_model.load_state_dict(torch.load(latest_ckpt, map_location=device, weights_only=False))
            print(f"Resumed successfully from round {start_round}")
            history_path = os.path.join(checkpoint_dir, "fedprox_round_history.csv")
            if os.path.exists(history_path):
                existing = pd.read_csv(history_path)
                for r in sorted(existing["round"].unique()):
                    round_history.append(
                        existing[existing["round"] == r].set_index("hospital")[
                            ["train_loss", "test_loss", "micro_f1", "precision", "recall",
                             "macro_f1", "macro_precision", "macro_recall"]
                        ].to_dict("index")
                    )
        except Exception as e:
            print(f"Checkpoint failed to load ({e}) — starting fresh from round 1 instead")
            start_round = 0
            round_history = []
    else:
        print("No FedProx checkpoint found — starting fresh from round 1")

    if start_round >= num_rounds:
        print(f"FedProx training already complete at round {start_round}.")
        return global_model, round_history

    for round_num in range(start_round + 1, num_rounds + 1):
        print(f"\n{'='*50}\nFEDPROX ROUND {round_num}/{num_rounds}\n{'='*50}")

        client_state_dicts = []
        client_sizes = []
        round_metrics = {}

        global_weights = copy.deepcopy(global_model.state_dict())

        for hospital in FEDERATED_CLIENTS:
            local_model = ResNet1D34(in_channels=12, num_classes=12).to(device)
            set_model_parameters(local_model, global_weights)

            train_loss = train_one_client_fedprox(
                local_model, global_weights, client_loaders[hospital]["train"], device,
                epochs=local_epochs, mu=mu
            )
            test_loss, f1, precision, recall, macro_f1, macro_precision, macro_recall = evaluate_one_client(
                local_model, client_loaders[hospital]["test"], device
            )

            print(f"  [{hospital}] train_loss={train_loss:.4f} test_loss={test_loss:.4f} "
                  f"micro_f1={f1:.4f} precision={precision:.4f} recall={recall:.4f} macro_f1: {macro_f1:.4f}, macro_precision: {macro_precision:.4f}, macro_recall: {macro_recall:.4f}")

            round_metrics[hospital] = {
                "train_loss": train_loss, "test_loss": test_loss,
                "micro_f1": f1, "precision": precision, "recall": recall,"macro_f1": macro_f1, "macro_precision": macro_precision, "macro_recall": macro_recall
            }
            client_state_dicts.append(get_model_parameters(local_model))
            client_sizes.append(client_loaders[hospital]["train_size"])

        new_global_weights = federated_average(client_state_dicts, client_sizes)
        set_model_parameters(global_model, new_global_weights)
        round_history.append(round_metrics)
        print(f"\nFedProx Round {round_num} complete. Global model updated.")

        if round_num % checkpoint_every == 0 or round_num == num_rounds:
            ckpt_path = os.path.join(checkpoint_dir, f"fedprox_global_model_round{round_num}.pt")
            torch.save(global_model.state_dict(), ckpt_path)

            history_rows = []
            for r, metrics in enumerate(round_history, start=1):
                for hospital, vals in metrics.items():
                    history_rows.append({"round": r, "hospital": hospital, **vals})
            pd.DataFrame(history_rows).to_csv(
                os.path.join(checkpoint_dir, "fedprox_round_history.csv"), index=False
            )
            print(f"  FedProx checkpoint + history saved at round {round_num}")

    return global_model, round_history


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    client_loaders = build_client_loaders(metadata_df, FEDERATED_CLIENTS, batch_size=BATCH_SIZE)

    global_model, round_history = run_fedprox_training(client_loaders, device)

    print("\n\nFedProx training run complete (30 rounds).")
# ============ END: fedprox.py ============

Using device: cuda
chapman_shaoxing: 7522 train / 1885 test
cpsc_2018: 4794 train / 1200 test
georgia: 6520 train / 1641 test
ningbo: 22246 train / 5594 test
ptb-xl: 16829 train / 4203 test
Found checkpoint: C:\ECG_Project\training\fedprox_global_model_round15.pt (round 15) — attempting to load
Resumed successfully from round 15

FEDPROX ROUND 16/30
  [chapman_shaoxing] train_loss=0.1240 test_loss=0.1112 micro_f1=0.8295 precision=0.8284 recall=0.8306 macro_f1: 0.5445, macro_precision: 0.6077, macro_recall: 0.5690
  [cpsc_2018] train_loss=0.1316 test_loss=0.1055 micro_f1=0.7690 precision=0.8601 recall=0.6954 macro_f1: 0.4017, macro_precision: 0.5192, macro_recall: 0.3861
  [georgia] train_loss=0.1989 test_loss=0.1743 micro_f1=0.6946 precision=0.7648 recall=0.6361 macro_f1: 0.5181, macro_precision: 0.5667, macro_recall: 0.4895
  [ningbo] train_loss=0.1086 test_loss=0.1145 micro_f1=0.8038 precision=0.8608 recall=0.7539 macro_f1: 0.3663, macro_precision: 0.4374, macro_recall: 0.3430
  [ptb

In [20]:
# ============ START: fedopt.py ============
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

NUM_ROUNDS = 30
LOCAL_EPOCHS = 1
BATCH_SIZE = 16
CHECKPOINT_EVERY = 5

# FedAdam server-side hyperparameters (standard defaults, Reddi et al. 2020)
SERVER_LR = 0.01
BETA1 = 0.9
BETA2 = 0.99
EPSILON = 1e-3


def fedavg_aggregate(client_state_dicts, client_sizes):
    """Standard FedAvg weighted average -- same as before, used as the RAW aggregation
    before FedAdam's momentum adjustment is applied on top."""
    total_size = sum(client_sizes)
    avg_weights = OrderedDict()
    for key in client_state_dicts[0].keys():
        weighted_sum = sum(
            client_state_dicts[i][key].float() * (client_sizes[i] / total_size)
            for i in range(len(client_state_dicts))
        )
        avg_weights[key] = weighted_sum
    return avg_weights


def fedadam_update(global_weights, avg_weights, m, v, server_lr=SERVER_LR, beta1=BETA1, beta2=BETA2, eps=EPSILON):
    """
    FedAdam server-side update. Treats (avg_weights - global_weights) as a
    'pseudo-gradient' from the server's perspective, then applies Adam-style
    momentum + adaptive scaling on top of it -- same idea as Adam for local
    training, just applied to the AGGREGATION step instead.
    """
    new_global = OrderedDict()
    new_m = OrderedDict()
    new_v = OrderedDict()

    for key in global_weights.keys():
        delta = avg_weights[key].float() - global_weights[key].float()

        new_m[key] = beta1 * m[key] + (1 - beta1) * delta
        new_v[key] = beta2 * v[key] + (1 - beta2) * (delta ** 2)

        new_global[key] = global_weights[key].float() + server_lr * new_m[key] / (torch.sqrt(new_v[key]) + eps)

    return new_global, new_m, new_v


def find_latest_checkpoint(checkpoint_dir):
    pattern = os.path.join(checkpoint_dir, "fedopt_global_model_round*.pt")
    checkpoints = glob.glob(pattern)
    if not checkpoints:
        return None, 0
    rounds = [int(f.split("round")[-1].split(".pt")[0]) for f in checkpoints]
    latest_round = max(rounds)
    latest_path = os.path.join(checkpoint_dir, f"fedopt_global_model_round{latest_round}.pt")
    try:
        torch.load(latest_path, map_location="cpu", weights_only=False)
        return latest_path, latest_round
    except Exception as e:
        print(f"Checkpoint failed to load ({e}) — starting fresh")
        return None, 0


def run_fedopt_training(client_loaders, device, num_rounds=NUM_ROUNDS, local_epochs=LOCAL_EPOCHS,
                         checkpoint_dir=CHECKPOINT_DIR, checkpoint_every=CHECKPOINT_EVERY):
    global_model = ResNet1D34(in_channels=12, num_classes=12).to(device)

    latest_ckpt, start_round = find_latest_checkpoint(checkpoint_dir)
    round_history = []

    # Initialize server-side momentum/variance accumulators at zero
    m = OrderedDict((k, torch.zeros_like(v).float()) for k, v in global_model.state_dict().items())
    v_acc = OrderedDict((k, torch.zeros_like(val).float()) for k, val in global_model.state_dict().items())

    if latest_ckpt:
        print(f"Resuming FedOpt from checkpoint: {latest_ckpt} (round {start_round})")
        global_model.load_state_dict(torch.load(latest_ckpt, map_location=device, weights_only=False))
        history_path = os.path.join(checkpoint_dir, "fedopt_round_history.csv")
        if os.path.exists(history_path):
            existing = pd.read_csv(history_path)
            for r in sorted(existing["round"].unique()):
                round_history.append(
                    existing[existing["round"] == r].set_index("hospital")[
                        ["train_loss", "test_loss", "micro_f1", "precision", "recall",
                         "macro_f1", "macro_precision", "macro_recall"]
                    ].to_dict("index")
                )
        # NOTE: m/v accumulators reset to zero on resume -- a minor approximation,
        # acceptable since momentum re-warms quickly over a few rounds
    else:
        print("No FedOpt checkpoint found — starting fresh from round 1")

    if start_round >= num_rounds:
        print(f"FedOpt training already complete at round {start_round}.")
        return global_model, round_history

    for round_num in range(start_round + 1, num_rounds + 1):
        print(f"\n{'='*50}\nFEDOPT ROUND {round_num}/{num_rounds}\n{'='*50}")

        client_state_dicts = []
        client_sizes = []
        round_metrics = {}

        global_weights = copy.deepcopy(global_model.state_dict())

        for hospital in FEDERATED_CLIENTS:
            local_model = ResNet1D34(in_channels=12, num_classes=12).to(device)
            set_model_parameters(local_model, global_weights)

            # Local training is PLAIN FedAvg-style (no proximal term) -- FedOpt's
            # novelty is entirely in the server aggregation step, not local training
            train_loss = train_one_client(
                local_model, client_loaders[hospital]["train"], device, epochs=local_epochs
            )
            test_loss, f1, precision, recall, macro_f1, macro_precision, macro_recall = evaluate_one_client(
                local_model, client_loaders[hospital]["test"], device
            )

            print(f"  [{hospital}] train_loss={train_loss:.4f} test_loss={test_loss:.4f} "
                  f"micro_f1={f1:.4f} macro_f1={macro_f1:.4f}")

            round_metrics[hospital] = {
                "train_loss": train_loss, "test_loss": test_loss,
                "micro_f1": f1, "precision": precision, "recall": recall,
                "macro_f1": macro_f1, "macro_precision": macro_precision, "macro_recall": macro_recall
            }
            client_state_dicts.append(get_model_parameters(local_model))
            client_sizes.append(client_loaders[hospital]["train_size"])

        # Step 1: standard FedAvg aggregation
        avg_weights = fedavg_aggregate(client_state_dicts, client_sizes)
        # Step 2: FedAdam momentum adjustment ON TOP of the averaged weights
        new_global_weights, m, v_acc = fedadam_update(global_weights, avg_weights, m, v_acc)

        set_model_parameters(global_model, new_global_weights)
        round_history.append(round_metrics)
        print(f"\nFedOpt Round {round_num} complete. Global model updated.")

        if round_num % checkpoint_every == 0 or round_num == num_rounds:
            ckpt_path = os.path.join(checkpoint_dir, f"fedopt_global_model_round{round_num}.pt")
            torch.save(global_model.state_dict(), ckpt_path)

            history_rows = []
            for r, metrics in enumerate(round_history, start=1):
                for hospital, vals in metrics.items():
                    history_rows.append({"round": r, "hospital": hospital, **vals})
            pd.DataFrame(history_rows).to_csv(
                os.path.join(checkpoint_dir, "fedopt_round_history.csv"), index=False
            )
            print(f"  FedOpt checkpoint + history saved at round {round_num}")

    return global_model, round_history


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    client_loaders = build_client_loaders(metadata_df, FEDERATED_CLIENTS, batch_size=BATCH_SIZE)

    global_model, round_history = run_fedopt_training(client_loaders, device)
    print("\n\nFedOpt training run complete.")
# ============ END: fedopt.py ============

Using device: cuda
chapman_shaoxing: 7522 train / 1885 test
cpsc_2018: 4794 train / 1200 test
georgia: 6520 train / 1641 test
ningbo: 22246 train / 5594 test
ptb-xl: 16829 train / 4203 test
No FedOpt checkpoint found — starting fresh from round 1

FEDOPT ROUND 1/30
  [chapman_shaoxing] train_loss=0.1694 test_loss=0.1342 micro_f1=0.7665 macro_f1=0.3652
  [cpsc_2018] train_loss=0.1758 test_loss=0.1495 micro_f1=0.6119 macro_f1=0.2250
  [georgia] train_loss=0.2426 test_loss=0.2457 micro_f1=0.5456 macro_f1=0.2752
  [ningbo] train_loss=0.1208 test_loss=0.1100 micro_f1=0.8083 macro_f1=0.3376
  [ptb-xl] train_loss=0.1579 test_loss=0.1371 micro_f1=0.8015 macro_f1=0.4064

FedOpt Round 1 complete. Global model updated.

FEDOPT ROUND 2/30
  [ningbo] train_loss=0.1159 test_loss=0.1079 micro_f1=0.8083 macro_f1=0.4002
  [ptb-xl] train_loss=0.1352 test_loss=0.1287 micro_f1=0.7976 macro_f1=0.4180

FedOpt Round 2 complete. Global model updated.

FEDOPT ROUND 3/30
  [chapman_shaoxing] train_loss=0.1270 t

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_per_class_f1(model, test_loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for signals, labels in test_loader:
            signals = signals.to(device)
            outputs = model(signals)
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    return dict(zip(FINAL_CLASSES, per_class_f1))


def compute_f1_std_table(checkpoint_path, method_name):
    model = ResNet1D34(in_channels=12, num_classes=12).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=False))

    rows = []
    for hospital in FEDERATED_CLIENTS:
        per_class = get_per_class_f1(model, client_loaders[hospital]["test"], device)
        f1_values = list(per_class.values())

        row = {"method": method_name, "hospital": hospital}
        row.update(per_class)
        row["macro_f1_check"] = np.mean(f1_values)   # sanity cross-check vs. your existing macro_f1
        row["f1_std"] = np.std(f1_values)             # FedCVD's exact metric
        rows.append(row)

    return pd.DataFrame(rows)


fedavg_std_df = compute_f1_std_table(r"C:\ECG_Project\training\fedavg\global_model_round30.pt", "FedAvg")
fedprox_std_df = compute_f1_std_table(r"C:\ECG_Project\training\fedprox_mu0.001\fedprox_global_model_round30.pt", "FedProx")
fedopt_std_df = compute_f1_std_table(r"C:\ECG_Project\training\fedopt\fedopt_global_model_round30.pt", "FedOpt")

f1_std_combined = pd.concat([fedavg_std_df, fedprox_std_df, fedopt_std_df], ignore_index=True)
f1_std_combined.to_csv(r"C:\ECG_Project\training\f1_std_comparison.csv", index=False)

print(f1_std_combined[["method", "hospital", "f1_std", "macro_f1_check"]])

     method          hospital    f1_std  macro_f1_check
0    FedAvg  chapman_shaoxing  0.261030        0.619287
1    FedAvg         cpsc_2018  0.313318        0.343060
2    FedAvg           georgia  0.299359        0.533119
3    FedAvg            ningbo  0.320404        0.524520
4    FedAvg            ptb-xl  0.324308        0.466186
5   FedProx  chapman_shaoxing  0.371398        0.405185
6   FedProx         cpsc_2018  0.262659        0.231276
7   FedProx           georgia  0.350983        0.369915
8   FedProx            ningbo  0.378630        0.373261
9   FedProx            ptb-xl  0.327884        0.302054
10   FedOpt  chapman_shaoxing  0.088949        0.035587
11   FedOpt         cpsc_2018  0.088762        0.039487
12   FedOpt           georgia  0.100901        0.045421
13   FedOpt            ningbo  0.100717        0.039658
14   FedOpt            ptb-xl  0.253837        0.086404


In [ ]:
#==============CENTRALIZED TRAINING================
NUM_EPOCHS = 30          # matches FedAvg/FedProx's 30-round budget for fair comparison
BATCH_SIZE = 16
CHECKPOINT_EVERY = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

metadata_df = pd.read_csv(SPLIT_CSV_PATH)

# Build individual per-hospital datasets (reuses your existing ECGDataset class),
# then pool them into ONE dataset — hospital identity is discarded here on purpose
train_datasets = [ECGDataset(metadata_df, PROCESSED_DIR, h, "train") for h in FEDERATED_CLIENTS]
test_datasets = [ECGDataset(metadata_df, PROCESSED_DIR, h, "test") for h in FEDERATED_CLIENTS]

pooled_train = ConcatDataset(train_datasets)
pooled_test = ConcatDataset(test_datasets)

pooled_train_loader = DataLoader(pooled_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
pooled_test_loader = DataLoader(pooled_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Pooled train size: {len(pooled_train)}  |  Pooled test size: {len(pooled_test)}")


def evaluate_central(model, test_loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for signals, labels in test_loader:
            signals, labels = signals.to(device), labels.to(device)
            outputs = model(signals)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    avg_loss = total_loss / len(test_loader)
    micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, micro_f1, macro_f1


def find_latest_central_checkpoint(checkpoint_dir):
    import glob
    pattern = os.path.join(checkpoint_dir, "central_model_epoch*.pt")
    checkpoints = glob.glob(pattern)
    if not checkpoints:
        return None, 0
    epochs = [int(f.split("epoch")[-1].split(".pt")[0]) for f in checkpoints]
    latest_epoch = max(epochs)
    latest_path = os.path.join(checkpoint_dir, f"central_model_epoch{latest_epoch}.pt")
    try:
        torch.load(latest_path, map_location="cpu", weights_only=False)
        return latest_path, latest_epoch
    except Exception as e:
        print(f"Checkpoint failed to load ({e}) — starting fresh")
        return None, 0


def run_central_training():
    model = ResNet1D34(in_channels=12, num_classes=12).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    latest_ckpt, start_epoch = find_latest_central_checkpoint(CHECKPOINT_DIR)
    history = []

    if latest_ckpt:
        print(f"Resuming Central from checkpoint: {latest_ckpt} (epoch {start_epoch})")
        model.load_state_dict(torch.load(latest_ckpt, map_location=device, weights_only=False))
        history_path = os.path.join(CHECKPOINT_DIR, "central_history.csv")
        if os.path.exists(history_path):
            history = pd.read_csv(history_path).to_dict("records")
    else:
        print("No Central checkpoint found — starting fresh from epoch 1")

    if start_epoch >= NUM_EPOCHS:
        print(f"Central training already complete at epoch {start_epoch}.")
        return model, history

    for epoch in range(start_epoch + 1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        n_batches = 0

        for signals, labels in pooled_train_loader:
            signals, labels = signals.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(signals)
            loss = criterion(outputs, labels)

            if not torch.isfinite(loss):
                print("  Skipping a batch with non-finite loss")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        train_loss = total_loss / max(n_batches, 1)
        test_loss, micro_f1, macro_f1 = evaluate_central(model, pooled_test_loader, device)

        print(f"Epoch {epoch}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  test_loss={test_loss:.4f}  "
              f"micro_f1={micro_f1:.4f}  macro_f1={macro_f1:.4f}")

        history.append({"epoch": epoch, "train_loss": train_loss, "test_loss": test_loss,
                         "micro_f1": micro_f1, "macro_f1": macro_f1})

        if epoch % CHECKPOINT_EVERY == 0 or epoch == NUM_EPOCHS:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"central_model_epoch{epoch}.pt")
            torch.save(model.state_dict(), ckpt_path)
            pd.DataFrame(history).to_csv(os.path.join(CHECKPOINT_DIR, "central_history.csv"), index=False)
            print(f"  Checkpoint + history saved at epoch {epoch}")

    return model, history


if __name__ == "__main__":
    model, history = run_central_training()
    print("\nCentral training run complete.")

Using device: cuda
Pooled train size: 57911  |  Pooled test size: 14523
No Central checkpoint found — starting fresh from epoch 1
Epoch 1/30  train_loss=0.1642  test_loss=0.1475  micro_f1=0.7503  macro_f1=0.5134
Epoch 2/30  train_loss=0.1310  test_loss=0.1288  micro_f1=0.7693  macro_f1=0.5459
Epoch 3/30  train_loss=0.1221  test_loss=0.1195  micro_f1=0.7732  macro_f1=0.5444
Epoch 4/30  train_loss=0.1151  test_loss=0.1199  micro_f1=0.7805  macro_f1=0.5652
Epoch 5/30  train_loss=0.1096  test_loss=0.1154  micro_f1=0.7911  macro_f1=0.5847
  Checkpoint + history saved at epoch 5
Epoch 6/30  train_loss=0.1053  test_loss=0.1140  micro_f1=0.7923  macro_f1=0.5988
Epoch 7/30  train_loss=0.1013  test_loss=0.1113  micro_f1=0.7987  macro_f1=0.6132
Epoch 8/30  train_loss=0.0981  test_loss=0.1106  micro_f1=0.7999  macro_f1=0.6014
Epoch 9/30  train_loss=0.0948  test_loss=0.1071  micro_f1=0.8063  macro_f1=0.6273
Epoch 10/30  train_loss=0.0919  test_loss=0.1102  micro_f1=0.8010  macro_f1=0.6282
  Checkpo

In [26]:
#============GRAD CAM==============
# >>> UPDATE THESE TWO PATHS to match your new folder locations exactly <
MODEL_CHECKPOINTS = {
    "fedavg":  os.path.join(CHECKPOINT_DIR, "fedavg", "global_model_round30.pt"),
    "fedprox": os.path.join(CHECKPOINT_DIR, "fedprox_mu0.001", "fedprox_global_model_round30.pt"),
    "fedopt":  os.path.join(CHECKPOINT_DIR, "fedopt", "fedopt_global_model_round30.pt"),
}

OUTPUT_DIR = os.path.join(CHECKPOINT_DIR, "stage10_gradcam")
os.makedirs(OUTPUT_DIR, exist_ok=True)
HEATMAP_NPZ_PATH = os.path.join(OUTPUT_DIR, "gradcam_heatmaps.npz")
SUMMARY_CSV_PATH = os.path.join(OUTPUT_DIR, "gradcam_summary.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MIN_RECORDS_PER_CELL = 5


class GradCAM1D:
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, x: torch.Tensor, target_class_idx: int, signal_length: int) -> np.ndarray:
        self.model.zero_grad(set_to_none=True)
        output = self.model(x)
        score = output[0, target_class_idx]
        score.backward()

        weights = self.gradients.mean(dim=2, keepdim=True)
        cam = (weights * self.activations).sum(dim=1)
        cam = F.relu(cam)

        cam = cam.unsqueeze(1)
        cam = F.interpolate(cam, size=signal_length, mode="linear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam)
        return cam


def load_model(checkpoint_path: str) -> torch.nn.Module:
    model = ResNet1D34(in_channels=12, num_classes=len(FINAL_CLASSES)).to(DEVICE)
    state_dict = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state_dict)
    model.eval()
    # NOTE: parameters are NOT frozen here — Grad-CAM's backward() requires an
    # intact autograd graph. eval() mode alone (disabling dropout/batchnorm
    # updates) is sufficient and correct; freezing requires_grad breaks backward().
    return model


def run_gradcam_for_model(model_name: str, checkpoint_path: str, metadata_df: pd.DataFrame):
    print(f"\n=== Grad-CAM: {model_name} ({checkpoint_path}) ===")
    model = load_model(checkpoint_path)
    cam_engine = GradCAM1D(model, target_layer=model.layer4)

    results = {}
    counts = {}
    summary_rows = []

    for hospital in FEDERATED_CLIENTS:
        test_ds = ECGDataset(metadata_df, PROCESSED_DIR, hospital, "test")
        if len(test_ds) == 0:
            print(f"  [{hospital}] no test records, skipping")
            continue

        loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

        for class_idx, class_name in enumerate(FINAL_CLASSES):
            key = f"{model_name}__{hospital}__{class_name}"
            heatmap_sum = np.zeros(5000, dtype=np.float64)
            n_used = 0

            for signal, label in loader:
                if label[0, class_idx].item() != 1.0:
                    continue

                signal = signal.to(DEVICE)
                cam = cam_engine.generate(signal, target_class_idx=class_idx, signal_length=signal.shape[-1])
                heatmap_sum += cam
                n_used += 1

            if n_used >= MIN_RECORDS_PER_CELL:
                results[key] = (heatmap_sum / n_used).astype(np.float32)
            counts[key] = n_used
            summary_rows.append({
                "model": model_name, "hospital": hospital, "class": class_name,
                "n_records_used": n_used,
                "included": n_used >= MIN_RECORDS_PER_CELL
            })
            if n_used > 0:
                print(f"  [{hospital}/{class_name}] n={n_used}"
                      + ("" if n_used >= MIN_RECORDS_PER_CELL else "  (below min, excluded)"))

    return results, summary_rows


if __name__ == "__main__":
    metadata_df = pd.read_csv(SPLIT_CSV_PATH)

    all_heatmaps = {}
    all_summary_rows = []

    for model_name, ckpt_path in MODEL_CHECKPOINTS.items():
        if not os.path.exists(ckpt_path):
            print(f"WARNING: checkpoint not found for {model_name}: {ckpt_path} — skipping")
            continue
        heatmaps, summary_rows = run_gradcam_for_model(model_name, ckpt_path, metadata_df)
        all_heatmaps.update(heatmaps)
        all_summary_rows.extend(summary_rows)

    np.savez_compressed(HEATMAP_NPZ_PATH, **all_heatmaps)
    pd.DataFrame(all_summary_rows).to_csv(SUMMARY_CSV_PATH, index=False)

    print(f"\nSaved {len(all_heatmaps)} heatmaps to: {HEATMAP_NPZ_PATH}")
    print(f"Saved summary to: {SUMMARY_CSV_PATH}")
# ============ END: gradcam.py ============


=== Grad-CAM: fedavg (C:\ECG_Project\training\fedavg\global_model_round30.pt) ===
  [chapman_shaoxing/AF] n=355
  [chapman_shaoxing/IAVB] n=50
  [chapman_shaoxing/LAD] n=76
  [chapman_shaoxing/LBBB] n=41
  [chapman_shaoxing/NSIVCB] n=47
  [chapman_shaoxing/NSR] n=364
  [chapman_shaoxing/PAC] n=53
  [chapman_shaoxing/QAb] n=47
  [chapman_shaoxing/RBBB] n=91
  [chapman_shaoxing/SB] n=776
  [chapman_shaoxing/STach] n=312
  [chapman_shaoxing/TAb] n=374
  [cpsc_2018/AF] n=275
  [cpsc_2018/IAVB] n=166
  [cpsc_2018/LBBB] n=55
  [cpsc_2018/NSIVCB] n=1  (below min, excluded)
  [cpsc_2018/NSR] n=184
  [cpsc_2018/PAC] n=138
  [cpsc_2018/RBBB] n=372
  [cpsc_2018/SB] n=9
  [cpsc_2018/STach] n=60
  [cpsc_2018/TAb] n=4  (below min, excluded)
  [georgia/AF] n=114
  [georgia/IAVB] n=154
  [georgia/LAD] n=188
  [georgia/LBBB] n=46
  [georgia/NSIVCB] n=41
  [georgia/NSR] n=350
  [georgia/PAC] n=128
  [georgia/QAb] n=93
  [georgia/RBBB] n=108
  [georgia/SB] n=335
  [georgia/STach] n=252
  [georgia/TAb] n